In [1]:
from llama_index.core import StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
#from llama_index.core.vector_stores.qdrant import QdrantVectorStore

from qdrant_client import QdrantClient
import qdrant_client
import os
from llama_parse import LlamaParse
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.node_parser import MarkdownElementNodeParser
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.retrievers import RecursiveRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

# 1. Setup Qdrant for metadata filtering

In [ ]:
client = QdrantClient(path="./qdrant_db")

vector_store = QdrantVectorStore(
    client=client,
    collection_name="mtc_collection"
)

In [3]:
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [8]:
parser = LlamaParse(
    api_key=os.environ['LLAMA_CLOUD_API_KEY'],
    result_type="markdown",
    parsing_instruction="""Extract MTC data with metadata tagging:
    
    For each document, identify and tag:
    - mtr_number: MTR/certificate number
    - heat_number: Heat/lot number
    - grade: Material grade (e.g., ASTM A182 F51)
    - order_number: Customer order number
    - date: Certificate date
    - material_spec: Full material specification
    
    For each table:
    - table_type: "chemical_analysis" or "mechanical_properties"
    - Preserve all rows with their values"""
)

In [9]:
documents = SimpleDirectoryReader(
    input_dir="./mtc_documents/",
    file_extractor={".pdf": parser}
).load_data()

2025-11-19 09:01:17,095 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"


Started parsing the file under job_id 7d162ff3-b5e3-4918-bcd3-fa4e04fc5651


2025-11-19 09:01:18,385 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7d162ff3-b5e3-4918-bcd3-fa4e04fc5651 "HTTP/1.1 200 OK"
2025-11-19 09:01:20,683 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7d162ff3-b5e3-4918-bcd3-fa4e04fc5651 "HTTP/1.1 200 OK"
2025-11-19 09:01:23,970 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7d162ff3-b5e3-4918-bcd3-fa4e04fc5651 "HTTP/1.1 200 OK"
2025-11-19 09:01:28,321 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7d162ff3-b5e3-4918-bcd3-fa4e04fc5651 "HTTP/1.1 200 OK"
2025-11-19 09:01:34,042 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7d162ff3-b5e3-4918-bcd3-fa4e04fc5651 "HTTP/1.1 200 OK"
2025-11-19 09:01:40,029 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7d162ff3-b5e3-4918-bcd3-fa4e04fc5651 "HTTP/1.1 200 OK"
2025-11-19 09:01:46,254 - INFO - HTTP Request: GET https://api.cloud.llamain

In [14]:
import re
def extract_mtr_number(text):
    match = re.search(r'MTR NO[.:]?\s*(\d+/\d+/\d+)', text, re.IGNORECASE)
    return match.group(1) if match else None

def extract_heat_number(text):
    match = re.search(r'HEAT NUMBER[:\s]+(\d+)', text, re.IGNORECASE)
    return match.group(1) if match else None

def extract_grade(text):
    match = re.search(r'ASTM A\d+\s+(?:UNS\s+)?([A-Z0-9]+)', text)
    return match.group(0) if match else None

def extract_date(text):
    match = re.search(r'\d{2}\.\d{2}\.\d{4}', text)
    return match.group(0) if match else None

In [15]:
enhanced_nodes = []
for doc in documents:
    # Extract metadata from document
    text = doc.text
    
    metadata = {
        "source": doc.metadata.get("file_name", ""),
        "mtr_number": extract_mtr_number(text),
        "heat_number": extract_heat_number(text),
        "grade": extract_grade(text),
        "date": extract_date(text)
    }
    
    # Parse into nodes with metadata
    node_parser = MarkdownElementNodeParser(llm=OpenAI(model="gpt-4o"))
    nodes = node_parser.get_nodes_from_documents([doc])
    
    # Add metadata to all nodes
    for node in nodes:
        node.metadata.update(metadata)
        enhanced_nodes.append(node)


0it [00:00, ?it/s]
0it [00:00, ?it/s]


In [18]:
from dotenv import load_dotenv 
load_dotenv()
index = VectorStoreIndex(
    enhanced_nodes,
    storage_context=storage_context,embed_model=OpenAIEmbedding(api_key=os.environ['OPEN_API_KEY'])
)


c:\Users\hahtsham\work\agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-19 09:06:40,682 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
c:\Users\hahtsham\work\agent\Lib\site-packages\llama_index\vector_stores\qdrant\base.py:852: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self._client.create_payload_index(


In [23]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter

filters = MetadataFilters(
    filters=[
        ExactMatchFilter(key="grade", value="ASTM E10")
    ]
)

query_engine = index.as_query_engine(
    similarity_top_k=5,llm=OpenAI(model="gpt-4o",api_key=os.environ['OPEN_API_KEY']),
    filters=filters
)

response = query_engine.query(
    "Tell me about this grade."
)
print(response)

2025-11-19 09:10:10,326 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Empty Response


In [36]:
from llama_index.core.schema import IndexNode

def create_table_summary_nodes(documents):
    """Create summary nodes for each table to improve retrieval"""
    
    all_nodes = []
    
    for doc in documents:
        # Parse document
        parser = MarkdownElementNodeParser(llm=OpenAI(model="gpt-4o"))
        base_nodes, table_objects = parser.get_nodes_and_objects([doc])
        
        # For each table, create a summary node
        for table_obj in table_objects:
            # Generate summary using LLM
            summary_prompt = f"""Summarize this table from an MTC document:

{table_obj.text}

Create a concise summary that includes:
1. What type of data this table contains
2. Key values and ranges
3. Material specifications if present
4. Any notable measurements or properties

Summary:"""
            
            llm = OpenAI(model="gpt-4o",api_key=os.environ['OPEN_API_KEY'])
            summary = llm.complete(summary_prompt).text
            
            # Create index node pointing to original table
            summary_node = IndexNode(
                text=summary,
                index_id=table_obj.node_id,
                metadata={
                    **table_obj.metadata,
                    "type": "table_summary",
                    "original_table_id": table_obj.node_id
                }
            )
            
            #all_nodes.append(summary_node)
            #all_nodes.append(table_obj)
        
        # Add text nodes
        all_nodes.extend(base_nodes)
        #print(base_nodes)
    
    return all_nodes

# Use in index creation
enriched_nodes = create_table_summary_nodes(documents)
index = VectorStoreIndex(enriched_nodes,embed_model=OpenAIEmbedding(api_key=os.environ['OPEN_API_KEY']))
query_engine=index.as_query_engine(llm=OpenAI(model="gpt-4o",api_key=os.environ['OPEN_API_KEY']))




2025-11-19 09:26:26,323 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [39]:
# Technical queries
queries = [
    "What is the Chemical analysis",
    "Can you summarize the Order No: 634728 Mechanical properties?"
]

for query in queries:
    response = query_engine.query(query)
    print(f"\nQ: {query}")
    print(f"A: {response}\n")
    print(f"Sources: {[node.metadata.get('file_name') for node in response.source_nodes]}")


2025-11-19 09:30:13,969 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-19 09:30:18,204 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Q: What is the Chemical analysis
A: The chemical analysis includes the following elements with their respective minimum and maximum values:

1. For the first document:
   - Carbon (C): Min 0.17, Max 0.030
   - Manganese (Mn): Min 0.20, Max 2.00
   - Silicon (Si): Min 0.08, Max 1.00
   - Phosphorus (P): Min 0.08, Max 0.030
   - Sulfur (S): Min 0.020, Max 0.020
   - Nickel (Ni): Min 6.50, Max 6.50
   - Molybdenum (Mo): Min 0.49, Max 0.49

2. For the second document:
   - Carbon (C): Min 0.15, Max 0.030
   - Silicon (Si): Min 0.20, Max 2.00
   - Manganese (Mn): Min 0.08, Max 1.00
   - Phosphorus (P): Min 0.00, Max 0.030
   - Sulfur (S): Min 0.00, Max 0.020
   - Chromium (Cr): Min 21.00, Max 23.00
   - Nickel (Ni): Min 4.50, Max 6.50
   - Molybdenum (Mo): Min 0.08, Max 0.20
   - Nitrogen (N): Min 0.00, Max 0.20

Sources: ['Flange MTC (First Case).pdf', 'Flange MTC (First Case).pdf']


2025-11-19 09:30:18,482 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-19 09:30:20,260 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Q: Can you summarize the Order No: 634728 Mechanical properties?
A: The mechanical properties for Order No: 634728 are as follows: Tensile Strength is 620 MPa, Yield Strength is 450 MPa, Elongation is 5 MIN, Reduction of Area is 21.00, and Hardness is 239 HBW.

Sources: ['Flange MTC (First Case).pdf', 'Flange MTC (First Case).pdf']


In [40]:
client.close()

In [ ]:
client = QdrantClient(path="./qdrant_db4")

vector_store = QdrantVectorStore(
    client=client,
    collection_name="mtc_collection1"
)



In [3]:
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [5]:
parser = LlamaParse(
    api_key=os.environ['LLAMA_CLOUD_API_KEY'],
    result_type="markdown",
    parsing_instruction="""Extract MTC data with metadata tagging:
    
    For each document, identify and tag:
    - mtr_number: MTR/certificate number
    - heat_number: Heat/lot number
    - grade: Material grade (e.g., ASTM A182 F51)
    - order_number: Customer order number
    - date: Certificate date
    - material_spec: Full material specification
    
    For each table:
    - table_type: "chemical_analysis" or "mechanical_properties"
    - Preserve all rows with their values"""
)
documents = SimpleDirectoryReader(
    input_dir="./mtc_documents/",
    file_extractor={".pdf": parser}
).load_data()

2025-11-19 10:20:49,804 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"


Started parsing the file under job_id 7f9994eb-abdc-4777-9073-68ca28dc0e91


2025-11-19 10:20:51,223 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7f9994eb-abdc-4777-9073-68ca28dc0e91 "HTTP/1.1 200 OK"
2025-11-19 10:20:53,763 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7f9994eb-abdc-4777-9073-68ca28dc0e91 "HTTP/1.1 200 OK"
2025-11-19 10:20:54,498 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7f9994eb-abdc-4777-9073-68ca28dc0e91/result/markdown "HTTP/1.1 200 OK"
